# Auxiliary Data Preprocessing and Merge

This notebook turns the findings from `auxiliary_data_investigation.ipynb` into reusable auxiliary features and joins the recommended features to the already-preprocessed train and test tables.

The final merged datasets use only the two high-priority auxiliary sources:

- HDB block metadata and coordinates;
- distance to and count of open MRT stations.

School, mall, COE, and stock features are still prepared for later ablation, but they are not included in the final HDB + MRT datasets. Existing auxiliary columns are removed from the processed base tables before the new merge, preventing duplicate or stale features.

No feature in this notebook uses `MONTHLY_RENT`. Raw and existing processed source files are never overwritten.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.neighbors import BallTree

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

EARTH_RADIUS_KM = 6_371.0088
SAVE_OUTPUTS = True


def locate_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "data/train.csv").exists() and (candidate / "data/auxiliary").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository data directory")


ROOT = locate_project_root()
DATA_DIR = ROOT / "data"
AUX_DIR = DATA_DIR / "auxiliary"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROCESSED_DIR / "auxiliary"

paths = {
    "train_raw": DATA_DIR / "train.csv",
    "test_raw": DATA_DIR / "test.csv",
    "train_processed": PROCESSED_DIR / "train_preprocessed.csv",
    "test_processed": PROCESSED_DIR / "test_preprocessed.csv",
    "processed_metadata": PROCESSED_DIR / "preprocessing_metadata.json",
    "hdb": AUX_DIR / "sg-hdb-block.csv",
    "mrt": AUX_DIR / "sg-mrt-stations.csv",
    "schools": AUX_DIR / "sg-schools.csv",
    "malls": AUX_DIR / "sg-shopping-malls.csv",
    "stocks": AUX_DIR / "sg-stock-prices.csv",
    "coe": AUX_DIR / "sg-coe-prices.csv",
}
missing_files = [str(input_path) for input_path in paths.values() if not input_path.exists()]
assert not missing_files, f"Missing input files: {missing_files}"

print(f"Project root: {ROOT}")
print("Final merge policy: processed core + HDB + open MRT")

## 1. Load and audit source tables

Identifiers such as postal codes are loaded as strings. The audit is descriptive and does not modify the sources.

In [ ]:
train_raw = pd.read_csv(paths["train_raw"])
test_raw = pd.read_csv(paths["test_raw"])
train_processed = pd.read_csv(paths["train_processed"], low_memory=False)
test_processed = pd.read_csv(paths["test_processed"], low_memory=False)
with paths["processed_metadata"].open(encoding="utf-8") as handle:
    processed_metadata = json.load(handle)

hdb_raw = pd.read_csv(paths["hdb"], dtype={"POSTAL_CODE": "string"})
mrt_raw = pd.read_csv(paths["mrt"])
schools_raw = pd.read_csv(paths["schools"], dtype={"POSTAL_CODE": "string"})
malls_raw = pd.read_csv(paths["malls"])
stocks_raw = pd.read_csv(paths["stocks"])
coe_raw = pd.read_csv(paths["coe"])

frames = {
    "train_raw": train_raw,
    "test_raw": test_raw,
    "train_processed": train_processed,
    "test_processed": test_processed,
    "hdb": hdb_raw,
    "mrt": mrt_raw,
    "schools": schools_raw,
    "malls": malls_raw,
    "stocks": stocks_raw,
    "coe": coe_raw,
}

source_audit = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_cells": int(frame.isna().sum().sum()),
    }
    for name, frame in frames.items()
]).set_index("dataset")
display(source_audit)

assert "MONTHLY_RENT" in train_processed.columns
assert "MONTHLY_RENT" not in test_processed.columns
assert len(train_processed) == len(train_raw)
assert len(test_processed) == len(test_raw)
assert train_processed["ROW_ID"].is_unique and test_processed["ROW_ID"].is_unique

## 2. Shared normalization helpers

Address keys use Unicode normalization, lowercase text, trimmed whitespace, and collapsed internal spaces. The same functions are applied to HDB, train, and test records.

In [ ]:
ADDRESS_KEYS = ["TOWN", "BLOCK", "STREET"]


def normalize_value(value):
    if pd.isna(value):
        return pd.NA
    value = unicodedata.normalize("NFKC", str(value)).strip().lower()
    return re.sub(r"\s+", " ", value)


def normalize_columns(frame, columns):
    result = frame.copy()
    for column in columns:
        result[column] = result[column].map(normalize_value).astype("string")
    return result


def parse_rent_month(values):
    text = values.astype("string").str.strip()
    parsed = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    for date_format in ("%Y-%m", "%b-%y"):
        missing = parsed.isna()
        parsed.loc[missing] = pd.to_datetime(text.loc[missing], format=date_format, errors="coerce")
    if parsed.isna().any():
        bad = text.loc[parsed.isna()].drop_duplicates().tolist()[:10]
        raise ValueError(f"Unrecognized rental month values: {bad}")
    return parsed.dt.to_period("M").dt.to_timestamp()


def clean_number(values):
    return pd.to_numeric(
        values.astype("string").str.replace(r"[$,]", "", regex=True).str.strip(),
        errors="coerce",
    )


def validate_coordinates(frame):
    latitude = pd.to_numeric(frame["LATITUDE"], errors="coerce")
    longitude = pd.to_numeric(frame["LONGITUDE"], errors="coerce")
    valid = latitude.between(-90, 90) & longitude.between(-180, 180)
    result = frame.loc[valid].copy()
    result["LATITUDE"] = latitude.loc[valid]
    result["LONGITUDE"] = longitude.loc[valid]
    return result

## 3. HDB block metadata

The investigation found complete address coverage and valid coordinates for all 9,464 blocks. Sentinel values such as `-1` and `NIL` are converted to missing values and accompanied by explicit quality flags.

In [ ]:
hdb = normalize_columns(hdb_raw, ADDRESS_KEYS)
hdb = validate_coordinates(hdb)

duplicate_hdb_keys = int(hdb.duplicated(ADDRESS_KEYS).sum())
if duplicate_hdb_keys:
    conflicting = hdb.groupby(ADDRESS_KEYS, dropna=False).nunique(dropna=False).max(axis=1).gt(1)
    assert not conflicting.any(), "Conflicting rows found for the same HDB address key"
    hdb = hdb.drop_duplicates(ADDRESS_KEYS, keep="first")

hdb["MAX_FLOOR"] = pd.to_numeric(hdb["MAX_FLOOR"], errors="coerce")
hdb["YEAR_COMPLETED"] = pd.to_numeric(hdb["YEAR_COMPLETED"], errors="coerce")
hdb["MAX_FLOOR_INVALID"] = hdb["MAX_FLOOR"].le(0).fillna(True).astype("int8")
hdb["YEAR_COMPLETED_INVALID"] = hdb["YEAR_COMPLETED"].le(0).fillna(True).astype("int8")
hdb.loc[hdb["MAX_FLOOR"].le(0), "MAX_FLOOR"] = np.nan
hdb.loc[hdb["YEAR_COMPLETED"].le(0), "YEAR_COMPLETED"] = np.nan

hdb["POSTAL_CODE"] = hdb["POSTAL_CODE"].str.strip().replace({"NIL": pd.NA, "nil": pd.NA})
hdb["POSTAL_CODE"] = hdb["POSTAL_CODE"].str.zfill(6)
hdb["POSTAL_SECTOR"] = hdb["POSTAL_CODE"].str[:2]
for column in ["SUBZONE", "PLANNING_AREA", "REGION"]:
    hdb[column] = hdb[column].map(normalize_value).astype("string")

hdb = hdb.rename(columns={
    "POSTAL_CODE": "HDB_POSTAL_CODE",
    "POSTAL_SECTOR": "HDB_POSTAL_SECTOR",
    "LATITUDE": "HDB_LATITUDE",
    "LONGITUDE": "HDB_LONGITUDE",
    "MAX_FLOOR": "HDB_MAX_FLOOR",
    "YEAR_COMPLETED": "HDB_YEAR_COMPLETED",
    "MAX_FLOOR_INVALID": "HDB_MAX_FLOOR_INVALID",
    "YEAR_COMPLETED_INVALID": "HDB_YEAR_COMPLETED_INVALID",
    "SUBZONE": "HDB_SUBZONE",
    "PLANNING_AREA": "HDB_PLANNING_AREA",
    "REGION": "HDB_REGION",
})

assert not hdb.duplicated(ADDRESS_KEYS).any()
assert hdb[["HDB_LATITUDE", "HDB_LONGITUDE"]].notna().all().all()

display(pd.Series({
    "unique_blocks": len(hdb),
    "duplicate_keys_removed": duplicate_hdb_keys,
    "missing_postal_codes": int(hdb["HDB_POSTAL_CODE"].isna().sum()),
    "invalid_max_floor": int(hdb["HDB_MAX_FLOOR_INVALID"].sum()),
    "invalid_year_completed": int(hdb["HDB_YEAR_COMPLETED_INVALID"].sum()),
}, name="HDB preprocessing"))

## 4. MRT, schools, and shopping malls

Only open MRT stations are used. Planned stations are excluded because the source has no opening dates. Duplicate point records are removed before spatial queries. School features are generated for both all schools and primary-level schools.

In [ ]:
def clean_point_table(frame, open_mrt_only=False):
    result = frame.copy()
    if open_mrt_only:
        status = result["STATUS"].map(normalize_value)
        result = result.loc[status.eq("open")].copy()
    result = validate_coordinates(result)
    result["NAME"] = result["NAME"].map(normalize_value).astype("string")
    return result.drop_duplicates(["NAME", "LATITUDE", "LONGITUDE"]).reset_index(drop=True)


mrt = clean_point_table(mrt_raw, open_mrt_only=True)
schools = clean_point_table(schools_raw)
malls = clean_point_table(malls_raw)

primary_mask = schools["MAINLEVEL_CODE"].astype("string").str.lower().str.contains(
    r"primary|p1", regex=True, na=False
)
primary_schools = schools.loc[primary_mask].copy()

amenity_inventory = pd.DataFrame({
    "dataset": ["open_mrt", "school", "primary_school", "mall"],
    "usable_locations": [len(mrt), len(schools), len(primary_schools), len(malls)],
}).set_index("dataset")
display(amenity_inventory)

In [ ]:
def spatial_features(origins, destinations, prefix):
    origin_coordinates = np.radians(
        origins[["HDB_LATITUDE", "HDB_LONGITUDE"]].to_numpy(dtype=float)
    )
    destination_coordinates = np.radians(
        destinations[["LATITUDE", "LONGITUDE"]].to_numpy(dtype=float)
    )
    tree = BallTree(destination_coordinates, metric="haversine")
    nearest_radians, _ = tree.query(origin_coordinates, k=1)
    return pd.DataFrame({
        f"DIST_TO_NEAREST_{prefix}_KM": (nearest_radians[:, 0] * EARTH_RADIUS_KM).astype("float32"),
        f"{prefix}_COUNT_WITHIN_500M": tree.query_radius(
            origin_coordinates, r=0.5 / EARTH_RADIUS_KM, count_only=True
        ).astype("int16"),
        f"{prefix}_COUNT_WITHIN_1KM": tree.query_radius(
            origin_coordinates, r=1.0 / EARTH_RADIUS_KM, count_only=True
        ).astype("int16"),
    }, index=origins.index)


block_features = hdb.copy()
for prefix, destinations in {
    "OPEN_MRT": mrt,
    "SCHOOL": schools,
    "PRIMARY_SCHOOL": primary_schools,
    "MALL": malls,
}.items():
    block_features = pd.concat(
        [block_features, spatial_features(block_features, destinations, prefix)],
        axis=1,
    )

distance_columns = [column for column in block_features if column.startswith("DIST_TO_NEAREST_")]
count_columns = [column for column in block_features if "_COUNT_WITHIN_" in column]
assert block_features[distance_columns + count_columns].notna().all().all()

display(block_features[distance_columns].describe(percentiles=[0.5, 0.9]).T[["mean", "50%", "90%", "max"]])

### Spatial-data limitation

The MRT table contains status but no historical opening date. An MRT station marked open in the supplied snapshot may not have been open for every 2021–2026 rental month. These features are still useful for a baseline, but this limitation should be recorded and checked through chronological validation.

## 5. Optional COE features

COE fields are cleaned by category and month. Auction rounds are aggregated, and the resulting table is shifted by one calendar month before it can be joined to rental records.

In [ ]:
coe = coe_raw.copy()
coe["CATEGORY"] = coe["CATEGORY"].map(normalize_value).str.upper()
coe["MONTH"] = pd.to_datetime(
    coe["YEAR"].astype(str) + "-" + coe["MONTH"].astype(str),
    format="%Y-%b",
    errors="raise",
).dt.to_period("M").dt.to_timestamp()

for column in ["QUOTA_PREMIUM", "PREVAILING_QUOTA_PREMIUM", "QUOTA", "BIDS_RECEIVED"]:
    coe[column] = clean_number(coe[column])

coe_monthly_long = coe.groupby(["MONTH", "CATEGORY"], as_index=False).agg(
    PREMIUM_MEAN=("QUOTA_PREMIUM", "mean"),
    PREVAILING_PREMIUM_MEAN=("PREVAILING_QUOTA_PREMIUM", "mean"),
    QUOTA_SUM=("QUOTA", "sum"),
    BIDS_SUM=("BIDS_RECEIVED", "sum"),
)
coe_monthly_long["BID_TO_QUOTA_RATIO"] = (
    coe_monthly_long["BIDS_SUM"] / coe_monthly_long["QUOTA_SUM"].replace(0, np.nan)
)

coe_monthly = coe_monthly_long.pivot(index="MONTH", columns="CATEGORY")
coe_monthly.columns = [f"COE_{category}_{metric}" for metric, category in coe_monthly.columns]
coe_monthly = coe_monthly.sort_index()
coe_monthly = coe_monthly.reindex(
    pd.date_range(coe_monthly.index.min(), coe_monthly.index.max(), freq="MS")
).shift(1)
coe_monthly.columns = [f"{column}_LAG1" for column in coe_monthly.columns]
coe_monthly.index.name = "RENT_DATE"
coe_monthly = coe_monthly.reset_index()

display(coe_monthly.tail(3))

## 6. Optional stock-market features

Raw price levels are not averaged across companies. Adjusted-close returns are calculated per symbol and summarized robustly by month. Daily changes above 50% in absolute value are excluded from the volatility calculation as likely corporate-action or data-quality artifacts. The monthly table is shifted by one month.

In [ ]:
stocks = stocks_raw.copy()
stocks["DATE"] = pd.to_datetime(stocks["DATE"], errors="coerce")
stocks["SYMBOL"] = stocks["SYMBOL"].astype("string")
stocks["ADJUSTED_CLOSE"] = pd.to_numeric(stocks["ADJUSTED_CLOSE"], errors="coerce")
stocks = stocks.dropna(subset=["DATE", "SYMBOL", "ADJUSTED_CLOSE"])
stocks = stocks.loc[stocks["ADJUSTED_CLOSE"].gt(0)].sort_values(["SYMBOL", "DATE"]).copy()
stocks["MONTH"] = stocks["DATE"].dt.to_period("M").dt.to_timestamp()

stocks["DAILY_RETURN"] = stocks.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].pct_change(fill_method=None)
stocks["DAILY_RETURN_CLEAN"] = stocks["DAILY_RETURN"].mask(stocks["DAILY_RETURN"].abs().gt(0.50))
stocks["NORMALIZED_LEVEL"] = (
    stocks["ADJUSTED_CLOSE"] /
    stocks.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].transform("first")
)

daily_volatility = stocks.groupby(["SYMBOL", "MONTH"], observed=True)["DAILY_RETURN_CLEAN"].std().rename("DAILY_VOLATILITY")
stock_month_end = stocks.groupby(["SYMBOL", "MONTH"], observed=True).tail(1).copy()
stock_month_end["MONTHLY_RETURN"] = stock_month_end.groupby("SYMBOL", observed=True)["ADJUSTED_CLOSE"].pct_change(fill_method=None)
stock_month_end = stock_month_end.merge(
    daily_volatility.reset_index(), on=["SYMBOL", "MONTH"], how="left", validate="one_to_one"
)


def positive_share(values):
    valid = values.dropna()
    return valid.gt(0).mean() if len(valid) else np.nan


stock_monthly = stock_month_end.groupby("MONTH", as_index=True, observed=True).agg(
    STOCK_INDEX_LEVEL_MEDIAN=("NORMALIZED_LEVEL", "median"),
    STOCK_RETURN_MEDIAN=("MONTHLY_RETURN", "median"),
    STOCK_RETURN_Q25=("MONTHLY_RETURN", lambda values: values.quantile(0.25)),
    STOCK_RETURN_Q75=("MONTHLY_RETURN", lambda values: values.quantile(0.75)),
    STOCK_POSITIVE_SHARE=("MONTHLY_RETURN", positive_share),
    STOCK_DAILY_VOLATILITY_MEDIAN=("DAILY_VOLATILITY", "median"),
    STOCK_ACTIVE_SYMBOL_COUNT=("SYMBOL", "nunique"),
    STOCK_RETURN_SYMBOL_COUNT=("MONTHLY_RETURN", "count"),
)
stock_monthly["STOCK_RETURN_IQR"] = stock_monthly["STOCK_RETURN_Q75"] - stock_monthly["STOCK_RETURN_Q25"]
stock_monthly = stock_monthly.drop(columns=["STOCK_RETURN_Q25", "STOCK_RETURN_Q75"])
stock_monthly = stock_monthly.reindex(
    pd.date_range(stock_monthly.index.min(), stock_monthly.index.max(), freq="MS")
).shift(1)
stock_monthly.columns = [f"{column}_LAG1" for column in stock_monthly.columns]
stock_monthly.index.name = "RENT_DATE"
stock_monthly = stock_monthly.reset_index()

display(stock_monthly.tail(3))

In [ ]:
market_features = coe_monthly.merge(
    stock_monthly, on="RENT_DATE", how="outer", validate="one_to_one"
).sort_values("RENT_DATE").reset_index(drop=True)

coe_feature_columns = [column for column in market_features if column.startswith("COE_")]
stock_feature_columns = [column for column in market_features if column.startswith("STOCK_")]
market_features["COE_DATA_AVAILABLE"] = market_features[coe_feature_columns].notna().any(axis=1).astype("int8")
market_features["STOCK_DATA_AVAILABLE"] = market_features[stock_feature_columns].notna().any(axis=1).astype("int8")

display(pd.DataFrame({
    "first_month": [market_features["RENT_DATE"].min()],
    "last_month": [market_features["RENT_DATE"].max()],
    "months": [len(market_features)],
    "coe_features": [len(coe_feature_columns)],
    "stock_features": [len(stock_feature_columns)],
}))

## 7. Build HDB + MRT transaction features

Following the auxiliary EDA recommendation, only HDB block fields and open-MRT proximity features are selected for the final merge. School, mall, COE, and stock features remain available for separate ablation experiments.

`ROW_ID` preserves the exact row order of the raw and processed train/test files. It is a join key, not a model feature.

In [ ]:
hdb_mrt_model_columns = [
    "HDB_LATITUDE", "HDB_LONGITUDE", "HDB_MAX_FLOOR", "HDB_YEAR_COMPLETED",
    "HDB_MAX_FLOOR_INVALID", "HDB_YEAR_COMPLETED_INVALID", "HDB_POSTAL_SECTOR",
    "HDB_SUBZONE", "HDB_PLANNING_AREA", "HDB_REGION",
    "DIST_TO_NEAREST_OPEN_MRT_KM",
    "OPEN_MRT_COUNT_WITHIN_500M", "OPEN_MRT_COUNT_WITHIN_1KM",
]
categorical_hdb_mrt_features = [
    "HDB_POSTAL_SECTOR", "HDB_SUBZONE", "HDB_PLANNING_AREA", "HDB_REGION",
]


def build_transaction_hdb_mrt_features(rental_data):
    keys = normalize_columns(rental_data[ADDRESS_KEYS], ADDRESS_KEYS)
    keys.insert(0, "ROW_ID", np.arange(len(keys), dtype=np.int64))
    keys["RENT_DATE"] = parse_rent_month(rental_data["RENT_APPROVAL_DATE"])

    feature_table = keys.merge(
        block_features[ADDRESS_KEYS + hdb_mrt_model_columns],
        on=ADDRESS_KEYS,
        how="left",
        validate="many_to_one",
        indicator="_HDB_MERGE",
    )
    feature_table["HDB_MATCHED"] = feature_table.pop("_HDB_MERGE").eq("both").astype("int8")

    building_age = feature_table["RENT_DATE"].dt.year - feature_table["HDB_YEAR_COMPLETED"]
    invalid_building_age = building_age.lt(0) | building_age.gt(100)
    feature_table["HDB_BUILDING_AGE_INVALID"] = invalid_building_age.fillna(False).astype("int8")
    feature_table["HDB_BUILDING_AGE_AT_RENT"] = building_age.mask(invalid_building_age).astype("float32")
    feature_table = feature_table.drop(columns=ADDRESS_KEYS)

    for column in categorical_hdb_mrt_features:
        feature_table[column] = feature_table[column].astype("string").fillna("__missing__")
    return feature_table.sort_values("ROW_ID").reset_index(drop=True)


train_hdb_mrt_features = build_transaction_hdb_mrt_features(train_raw)
test_hdb_mrt_features = build_transaction_hdb_mrt_features(test_raw)

train_join_rate = float(train_hdb_mrt_features["HDB_MATCHED"].mean())
test_join_rate = float(test_hdb_mrt_features["HDB_MATCHED"].mean())

assert len(train_hdb_mrt_features) == len(train_raw)
assert len(test_hdb_mrt_features) == len(test_raw)
assert train_hdb_mrt_features.columns.tolist() == test_hdb_mrt_features.columns.tolist()
assert train_hdb_mrt_features["ROW_ID"].tolist() == list(range(len(train_raw)))
assert test_hdb_mrt_features["ROW_ID"].tolist() == list(range(len(test_raw)))
assert min(train_join_rate, test_join_rate) >= 0.99
assert "MONTHLY_RENT" not in train_hdb_mrt_features.columns

display(pd.DataFrame({
    "rows": [len(train_hdb_mrt_features), len(test_hdb_mrt_features)],
    "columns": [train_hdb_mrt_features.shape[1], test_hdb_mrt_features.shape[1]],
    "HDB_join_rate": [train_join_rate, test_join_rate],
}, index=["train_hdb_mrt_features", "test_hdb_mrt_features"]))

## 8. Join with processed train and test

The existing processed tables already contain auxiliary fields from the earlier all-in-one preprocessing pipeline. Those fields are removed first. The newly prepared HDB + MRT feature matrices are then joined by `ROW_ID` with one-to-one validation.

The original `train_preprocessed.csv` and `test_preprocessed.csv` remain unchanged.

In [ ]:
EXISTING_AUXILIARY_PREFIXES = (
    "HDB_", "DIST_TO_NEAREST_", "MRT_COUNT_", "MALL_COUNT_",
    "PRIMARY_SCHOOL_COUNT_", "COE_", "STOCK_", "BUILDING_AGE_",
)


def existing_auxiliary_columns(frame):
    return [
        column for column in frame.columns
        if column.startswith(EXISTING_AUXILIARY_PREFIXES)
    ]


train_columns_to_remove = existing_auxiliary_columns(train_processed)
test_columns_to_remove = existing_auxiliary_columns(test_processed)
assert set(train_columns_to_remove) == set(test_columns_to_remove)

train_core_processed = train_processed.drop(columns=train_columns_to_remove)
test_core_processed = test_processed.drop(columns=test_columns_to_remove)

# RENT_DATE is already present in the processed base, so it is not duplicated.
train_aux_for_merge = train_hdb_mrt_features.drop(columns="RENT_DATE")
test_aux_for_merge = test_hdb_mrt_features.drop(columns="RENT_DATE")

train_preprocessed_hdb_mrt = train_core_processed.merge(
    train_aux_for_merge, on="ROW_ID", how="left", validate="one_to_one"
)
test_preprocessed_hdb_mrt = test_core_processed.merge(
    test_aux_for_merge, on="ROW_ID", how="left", validate="one_to_one"
)

target = processed_metadata.get("target", "MONTHLY_RENT")
test_column_order = test_preprocessed_hdb_mrt.columns.tolist()
train_preprocessed_hdb_mrt = train_preprocessed_hdb_mrt[test_column_order + [target]]

assert len(train_preprocessed_hdb_mrt) == len(train_processed)
assert len(test_preprocessed_hdb_mrt) == len(test_processed)
assert train_preprocessed_hdb_mrt.drop(columns=target).columns.tolist() == test_preprocessed_hdb_mrt.columns.tolist()
assert train_preprocessed_hdb_mrt["ROW_ID"].tolist() == train_processed["ROW_ID"].tolist()
assert test_preprocessed_hdb_mrt["ROW_ID"].tolist() == test_processed["ROW_ID"].tolist()
assert not any(column.endswith(("_x", "_y")) for column in train_preprocessed_hdb_mrt.columns)
assert not any(column.startswith(("COE_", "STOCK_")) for column in train_preprocessed_hdb_mrt.columns)
assert "DIST_TO_NEAREST_OPEN_MRT_KM" in train_preprocessed_hdb_mrt.columns
assert "DIST_TO_NEAREST_MALL_KM" not in train_preprocessed_hdb_mrt.columns
assert target not in test_preprocessed_hdb_mrt.columns

display(pd.DataFrame({
    "rows": [len(train_preprocessed_hdb_mrt), len(test_preprocessed_hdb_mrt)],
    "columns": [train_preprocessed_hdb_mrt.shape[1], test_preprocessed_hdb_mrt.shape[1]],
    "removed_old_auxiliary_columns": [len(train_columns_to_remove), len(test_columns_to_remove)],
    "added_HDB_MRT_columns": [len(train_aux_for_merge.columns) - 1, len(test_aux_for_merge.columns) - 1],
}, index=["train_preprocessed_hdb_mrt", "test_preprocessed_hdb_mrt"]))

## 9. Save HDB + MRT outputs

The row-level HDB + MRT matrices are saved for auditing. The main deliverables are the two fully merged processed datasets in `data/processed/`. Optional market and broader spatial tables remain separate and are not included in the main model tables.

In [ ]:
final_non_feature_columns = ["ROW_ID", "RENT_DATE"]
final_model_features = [
    column for column in test_preprocessed_hdb_mrt.columns
    if column not in final_non_feature_columns
]
base_categorical_features = [
    column for column in processed_metadata.get("categorical_features", [])
    if column not in train_columns_to_remove
]
final_categorical_features = list(dict.fromkeys(
    base_categorical_features + categorical_hdb_mrt_features
))

metadata = {
    "target": target,
    "row_id": "ROW_ID",
    "date_column": "RENT_DATE",
    "source_processed_files": ["train_preprocessed.csv", "test_preprocessed.csv"],
    "output_files": ["train_preprocessed_hdb_mrt.csv", "test_preprocessed_hdb_mrt.csv"],
    "hdb_join_rate": {"train": train_join_rate, "test": test_join_rate},
    "removed_existing_auxiliary_columns": train_columns_to_remove,
    "added_hdb_mrt_features": [
        column for column in train_hdb_mrt_features.columns
        if column not in ["ROW_ID", "RENT_DATE"]
    ],
    "model_features": final_model_features,
    "categorical_features": final_categorical_features,
    "excluded_by_auxiliary_eda": ["schools", "shopping_malls", "coe", "stocks"],
    "merge_policy": "remove existing auxiliary columns, then one-to-one join HDB + open MRT by ROW_ID",
}

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    block_features.to_csv(OUTPUT_DIR / "hdb_spatial_features.csv", index=False)
    market_features.to_csv(OUTPUT_DIR / "market_features_lag1.csv", index=False, date_format="%Y-%m-%d")
    train_hdb_mrt_features.to_csv(OUTPUT_DIR / "train_auxiliary_features.csv", index=False, date_format="%Y-%m-%d")
    test_hdb_mrt_features.to_csv(OUTPUT_DIR / "test_auxiliary_features.csv", index=False, date_format="%Y-%m-%d")
    train_preprocessed_hdb_mrt.to_csv(PROCESSED_DIR / "train_preprocessed_hdb_mrt.csv", index=False)
    test_preprocessed_hdb_mrt.to_csv(PROCESSED_DIR / "test_preprocessed_hdb_mrt.csv", index=False)
    with (OUTPUT_DIR / "auxiliary_preprocessing_metadata.json").open("w", encoding="utf-8") as handle:
        json.dump(metadata, handle, ensure_ascii=False, indent=2)

    saved_paths = [
        OUTPUT_DIR / "hdb_spatial_features.csv",
        OUTPUT_DIR / "market_features_lag1.csv",
        OUTPUT_DIR / "train_auxiliary_features.csv",
        OUTPUT_DIR / "test_auxiliary_features.csv",
        OUTPUT_DIR / "auxiliary_preprocessing_metadata.json",
        PROCESSED_DIR / "train_preprocessed_hdb_mrt.csv",
        PROCESSED_DIR / "test_preprocessed_hdb_mrt.csv",
    ]
    print("Saved files:")
    for output_path in saved_paths:
        print(f"- {output_path.relative_to(ROOT)}: {output_path.stat().st_size / 1_048_576:.2f} MiB")

## 10. Modeling handoff

- Use `train_preprocessed_hdb_mrt.csv` and `test_preprocessed_hdb_mrt.csv` as the matched modeling pair.
- Exclude `ROW_ID` and `RENT_DATE` from model inputs. Use `RENT_DATE` only for chronological validation splits.
- Start with this HDB + MRT dataset because these were the high-priority auxiliary sources in the investigation.
- Test school, mall, and lagged market features only as separate ablations; do not mix them into the baseline without future-period validation evidence.
- Use the categorical feature list from `auxiliary_preprocessing_metadata.json` for CatBoost.

The supplied MRT table lacks historical opening dates, so the MRT features represent the supplied network snapshot rather than a fully historical network.